In [6]:
import os
import json

In [7]:
unit_test = ""
error_message = ""
method_sig = ""
class_name = ""
full_fm = ""
other_method_sigs=""

In [8]:
zero_shot_prompt = f'''I need you to fix an error in a unit test, an error occurred while compiling and executing

The unit test is:

${unit_test}


The error chatMessage is:

${error_message}


The unit test is testing the method `${method_sig}` in the class `${class_name}`,
the source code of the method under test and its class is:

${full_fm}


<#if other_method_sigs?has_content>

The signatures of other methods in its class are `${other_method_sigs}`


Please fix the error and return the whole fixed unit test. You can use Junit 5 and Mockito 3. Adhere to Java 8 language style. No explanation is needed.'''

In [10]:
few_shot_prompt = f'''I need you to fix an error in a unit test. An error occurred while compiling or executing.

Below are a few examples of similar fixes. Follow the same style: minimal changes to make the test compile and run.

========================
EXAMPLE 1
========================
Original Unit Test:

```java
"public class Options_getOptionGroup_9_0_Test {{
    static class OptionGroup {{
        private final String id;
        OptionGroup(String id) {{
            this.id = id;
        }}
        
        String getId() {{
            return id;
        }}
    }}
    
    testGetOptionGroupReturnsNullWhenMissing() throws Exception {{
    Options options = new Options();
    Option opt = new Option(""b"", ""desc"");
    OptionGroup result = options.getOptionGroup(opt);
    assertNull(result, ""Expected null when no OptionGroup is associated with the option key"");
    
}}
```

Error chatMessage:
Error in Options_getOptionGroup_9_0_Test: incompatible types: org.apache.commons.cli.OptionGroup cannot be converted to org.apache.commons.cli.Options_getOptionGroup_9_0_Test.OptionGroup

Fixed Unit Test:

```java

"public class Options_getOptionGroup_9_0_Test {{
   
   testGetOptionGroupReturnsNullWhenMissing() throws Exception {{
    Options options = new Options();
    Option opt = new Option(""b"", ""desc"");
    OptionGroup result = options.getOptionGroup(opt);
    assertNull(result, ""Expected null when no OptionGroup is associated with the option key"");
}}
}}

```

========================
EXAMPLE 2
========================
Original Unit Test:

```java
INSERT BROKEN TEST HERE
```

Error chatMessage:
INSERT ERROR HERE

Fixed Unit Test:

```java
INSERT FIXED TEST HERE
```

========================
Now, fix the following.

The unit test is:

${unit_test}

The error chatMessage is:

${error_message}

The unit test is testing the method ${method_sig} in the class ${class_name},
the source code of the method under test and its class is:

${full_fm}

<#if other_method_sigs?has_content>
The signatures of other methods in its class are ${other_method_sigs}
</#if>

Please fix the error and return the whole fixed unit test. You can use Junit 5, and Mockito 3. Adhere to Java 8 language style. No explanation is needed.
Return only the fixed unit test.'''

In [11]:
cot_prompt = f'''I need you to fix an error in a unit test. An error occurred while compiling or executing.

The unit test is:

${unit_test}

The error chatMessage is:

${error_message}

The unit test is testing the method `${method_sig}` in the class `${class_name}`,
the source code of the method under test and its class is:

${full_fm}

<#if other_method_sigs?has_content>
The signatures of other methods in its class are `${other_method_sigs}`
</#if>

Chain-of-thought instruction:
# Procedures for Fixing the Unit Test:
Let's proceed step by step:

1. Pick out the statements that the errors occur internally, do not give them in the output.
2. Explain the causes of the errors internally, do not give them in the output.
3. Come up with solutions on how to fix the errors internally, do not give them in the output.
4. Provide the complete fixed unit test as the output, utilizing JUnit 5 and Mockito 3, and nothing else.

Constraints:
- Do NOT modify the production code.

Output Format:
< Generation Begin >
{{fixed unit test code only}}
< Generation Over >

Please fix the error and return the whole fixed unit test. You can use Junit 5, and Mockito 3. Adhere to Java 8 language style. No explanation is needed.
Return only the fixed unit test.'''

In [12]:
class_no = 9
method_no = 15

In [15]:
def class_data_path_fetcher(class_: str, method: str, class_map_path, class_info_dir): # returns path for where information about the class is stored
    with open(class_map_path, "r") as f:
        mapdata = json.load(f)
        
    classname = mapdata.get(class_).get("className")
    path_items = mapdata.get(class_).get("packageName").split(".")
    full_path = class_info_dir

    for i in path_items:
        full_path += f"/{i}"
    full_path += f"/{classname}"
    return full_path

In [16]:
def other_methods_signature_fetcher(class_info_fp) -> list:
    class_info_fp += "/class.json"
    with open(class_info_fp, "r") as g:
        classdata = json.load(g)
    
    other_methods = classdata.get("methodsBrief")

    return other_methods

In [23]:
def dep_class_path_fetcher(class_info_dir, class_info_fp, method: str): # takes in method number and class info path

    with open(class_info_fp+f"/{method}.json", "r") as f:
        method_info = json.load(f)
    
    dep_method_dict = method_info.get("dependentMethods")
    dep_class_path_list = []
    for key in dep_method_dict:
        path_split = key.split(".")
        fp = class_info_dir
        for i in path_split:
            fp += f"/{i}"
        dep_class_path_list.append(fp)

    return dep_class_path_list

In [18]:
class_data_path_fetcher("class9", "f", "/home/jaskeerat/paper1/cli_latest_stable/commons-cli/chatunitest-tmp/commons-cli/classMapping.json", "/home/jaskeerat/paper1/cli_latest_stable/commons-cli/chatunitest-tmp/commons-cli/class-info")

'/home/jaskeerat/paper1/cli_latest_stable/commons-cli/chatunitest-tmp/commons-cli/class-info/org/apache/commons/cli/help/TextHelpAppendable'

In [ ]:
other_methods_signature_fetcher('/home/jaskeerat/paper1/cli_latest_stable/commons-cli/chatunitest-tmp/commons-cli/class-info/org/apache/commons/cli/help/TextHelpAppendable')

In [24]:
dep_class_path_fetcher("/home/jaskeerat/paper1/cli_latest_stable/commons-cli/chatunitest-tmp/commons-cli/class-info", "/home/jaskeerat/paper1/cli_latest_stable/commons-cli/chatunitest-tmp/commons-cli/class-info/org/apache/commons/cli/help/OptionFormatter","10")

['/home/jaskeerat/paper1/cli_latest_stable/commons-cli/chatunitest-tmp/commons-cli/class-info/org/apache/commons/cli/help/OptionFormatter']